# فصل ۱۲: آشنایی با پایگاه داده‌های برداری (Vector Databases)
## بخش ۲: معماری پایگاه‌داده‌های برداری و الگوریتم‌های جستجوی تقریبی (ANN)

در درس‌نامه گذشته دیدیم که چرا دیتابیس‌های سنتی (مانند PostgreSQL یا MySQL) برای جستجوی معنایی مناسب نیستند و چرا جستجوی همه‌جانبه (Brute-Force) با افزایش تعداد داده‌ها به بن‌بست می‌رسد.

در این بخش، به بررسی عمیق **معماری داخلی وکتور دیتابیس‌ها**، تفاوت لایه ذخیره‌سازی و ایندکس‌گذاری، الگوریتم‌های محبوب **ANN** (شامل **IVF** و **HNSW**) و در نهایت **پیاده‌سازی عملی با ChromaDB در LangChain** می‌پردازیم.

### ۱. معماری وکتور دیتابیس‌ها: دو موتور در یک خودرو

یک پایگاه داده برداری مدرن (نظیر Chroma, Pinecone, Milvus, Qdrant) تنها یک فایل ساده نیست؛ بلکه از دو لایه مجزا و مکمل تشکیل شده است:

```
+-------------------------------------------------------------------------+
|                       Vector Database Architecture                      |
+-------------------------------------------------------------------------+
|                                                                         |
|  [1. لایه ایندکس و جستجو (Index Layer)]  <--- مستقر در RAM              |
|  - ساختار داده گراف یا درخت (HNSW / IVF)                                |
|  - نقشه ناوبری سریع در فضای برداری (مسیریابی در زمان کوئری)             |
|  - پیچیدگی زمانی جستجو: O(log N)                                        |
|                                                                         |
|-------------------------------------------------------------------------|
|                                                                         |
|  [2. لایه انبار داده (Storage Layer)]     <--- مستقر در Disk             |
|  - ذخیره متن اصلی سند (Raw Documents / Text Chunks)                     |
|  - ذخیره بردارهای خام به عنوان منبع اصلی (Raw Embeddings)                |
|  - ذخیره فراداده‌ها (Metadata: تاریخ، نویسنده، برچسب‌ها، شناسه)         |
|  - پایداری در برابر خاموش شدن سیستم (Persistence / ACID)                |
|                                                                         |
+-------------------------------------------------------------------------+
```

- **لایه ذخیره‌سازی (Storage Layer):** تضمین می‌کند که داده‌های شما با بسته شدن برنامه پاک نشوند. از سیستم‌های ذخیره‌سازی سنتی مانند SQLite، RocksDB یا سیستم‌فایل استفاده می‌کند.
- **لایه ایندکس (Index Layer):** ساختاری هوشمند در حافظه رم (RAM) می‌سازد تا بدون خواندن کل اطلاعات دیسک، در کسری از میلی‌ثانیه بردار سوال را به نزدیک‌ترین بردارهای مرتبط هدایت کند.

### ۲. تمثیل استادیوم: جستجوی دقیق (Exact) در برابر جستجوی تقریبی (ANN)

فرض کنید در یک **استادیوم فوتبال ۱۰۰,۰۰۰ نفری**، مأموری به دنبال تماشاگر خاصی می‌گردد:

1. **جستجوی دقیق (Exact kNN / Brute-Force):**
   - مأمور به تک‌تک صندلی‌ها از ردیف اول جایگاه ۱ تا ردیف آخر جایگاه ۱۰۰ مراجعه می‌کند و چهره همه را با عکس تطبیق می‌دهد.
   - **نتیجه:** دقت ۱۰۰٪ است، اما ساعت‌ها زمان می‌برد ($O(N)$).

2. **جستجوی تقریبی (Approximate Nearest Neighbors - ANN):**
   - تماشاگران با پیراهن‌های زرد در جایگاه جنوبی و پیراهن‌های قرمز در جایگاه شمالی نشسته‌اند. تابلوی راهنما (Index) نشان می‌دهد که فرد مورد نظر در جایگاه شمالی، بلوک ۴ قرار دارد.
   - مأمور مستقیماً به بلوک ۴ می‌رود و فرد را ظرف ۲ دقیقه پیدا می‌کند.
   - **معاوضه (Trade-off):** ممکن است فرد به اشتباه در صندلی کناری نشسته باشد (دقت ۹۹٪ به جای ۱۰۰٪)، اما سرعت جستجو **هزاران برابر** افزایش می‌یابد ($O(\log N)$).

### ۳. بررسی دو الگوریتم اصلی ساخت ایندکس: IVF در برابر HNSW

| ویژگی | الگوریتم IVF (Inverted File) | الگوریتم HNSW (Hierarchical Navigable Small World) |
| :--- | :--- | :--- |
| **روش کار** | خوشه‌بندی فضا به سلول‌های ورونوی (Voronoi) با $k$-Means | گراف چندلایه چندسطحی شبیه Skip-List و اتوبان‌ها |
| **عملکرد جستجو** | ابتدا نزدیک‌ترین سنترویدها پیدا می‌شوند، سپس داخل آن سلول‌ها جستجو می‌شود | از لایه بالا (پرش‌های بزرگ) به لایه پایین (کوچه‌ها و همسایه‌های دقیق) ناوبری می‌کند |
| **مصرف حافظه (RAM)** | بسیار کم و بهینه | بالاتر (به دلیل نگهداری یال‌های گراف در رم) |
| **سرعت و Recall** | خوب، اما در ابعاد بسیار بالا ممکن است مرزهای خوشه‌ها خطا ایجاد کند | فوق‌العاده سریع و دارای بالاترین نرخ یادآوری (Recall) |
| **کاربرد اصلی** | سیستم‌های بسیار عظیم با محدودیت RAM یا ابعاد متوسط | انتخاب پیش‌فرض وکتور دیتابیس‌های مدرن (Chroma, Pinecone, Weaviate) |

### ۴. مثلث مصالحه (The Trade-Off Triangle): قانون «ناهار مجانی وجود ندارد»

در مهندسی سیستم‌های جستجوی برداری، همیشه بین ۳ ضلع زیر یک مصالحه (Trade-off) برقرار است:

```
                         سرعت (Speed / Latency)
                                 /\
                                /  \
                               /    \
                              /      \
                             /        \
                            /  مثلث    \
                           /   مصالحه   \
                          /              \
                         /________________\
        دقت (Accuracy / Recall)         حافظه مصرفی (Memory / RAM)
```

- اگر **سرعت بالا + دقت بالا** می‌خواهید $\rightarrow$ باید هزینه حافظه RAM بیشتری بپردازید (مانند الگوریتم HNSW).
- اگر **حافظه RAM محدود** دارید $\rightarrow$ می‌توانید از کوانتیزاسیون (Quantization) یا الگوریتم IVF استفاده کنید که کمی سرعت یا Recall را کاهش می‌دهد.
- اگر **دقت مطلق ۱۰۰٪** بدون هیچ خطایی می‌خواهید $\rightarrow$ باید جستجوی کند Flat / Brute-Force را تحمل کنید.

### ۵. پیاده‌سازی عملی با ChromaDB و LangChain

دیتابیس **ChromaDB** یکی از محبوب‌ترین و سبک‌ترین پایگاه‌داده‌های برداری منبع‌باز است که به طور پیش‌فرض از الگوریتم **HNSW** برای ایندکس‌گذاری سریع استفاده می‌کند.

در سلول زیر محیط را تنظیم کرده و اتصالات مدل Embedding را بررسی می‌کنیم:

In [1]:
import os
import time
from dotenv import load_dotenv

# حذف پروکسی‌های مرده محیط سیستم
for var in ["http_proxy", "https_proxy", "HTTP_PROXY", "HTTPS_PROXY"]:
  os.environ.pop(var, None)

# فعال‌سازی حالت آفلاین برای استفاده فوری از مدل کش‌شده بدون تاخیر شبکه
os.environ["HF_HUB_OFFLINE"] = "1"
load_dotenv()

# راه‌اندازی مدل تعبیه متن محلی و چندزبانه HuggingFace
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print(
    "✅ مدل Embedding محلی (چندزبانه با پشتیبانی عالی از فارسی) با موفقیت"
    " بارگذاری شد."
)

d:\Project\LLM\llm-apps-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2612.28it/s]


✅ مدل Embedding محلی (چندزبانه با پشتیبانی عالی از فارسی) با موفقیت بارگذاری شد.


### ۶. ساخت اسناد نمونه همراه با فراداده‌ها (Metadata)

در کاربردهای واقعی، ما نه تنها متن (Document)، بلکه اطلاعات تکمیلی نظیر دسته‌بندی موضوعی، نویسنده، سال انتشار و غیره را نیز به عنوان فراداده ثبت می‌کنیم تا امکان **فیلترینگ ترکیبی (Hybrid / Metadata Filtering)** فراهم شود.

In [2]:
documents = [
    "هوش مصنوعی و یادگیری ماشین شاخه‌ای از علوم کامپیوتر هستند که سیستم‌ها را قادر می‌سازند از داده‌ها بیاموزند.",
    "شبکه‌های عصبی عمیق با لایه‌های پنهان متعدد برای پردازش زبان طبیعی و بینایی ماشین استفاده می‌شوند.",
    "درختان بلوط و راش از گونه‌های مهم جنگل‌های هیرکانی در شمال ایران هستند.",
    "گیاهان از طریق فتوسنتز، دی‌اکسید کربن هوا را جذب کرده و اکسیژن و قند تولید می‌کنند.",
    "تیم ملی فوتبال آرژانتین با درخشش لیونل مسی قهرمان جام جهانی فوتبال در قطر شد.",
    "ورزش روزانه و تمرینات هوازی سلامت قلب و عروق و طول عمر انسان را بهبود می‌بخشند.",
    "الگوریتم‌های رمزنگاری نامتقارن مانند RSA امنیت انتقال اطلاعات در بستر اینترنت را تضمین می‌کنند.",
    "پایگاه داده‌های برداری با بهره‌گیری از الگوریتم HNSW جستجوی شباهت را با سرعت O(log N) انجام می‌دهند."
]

metadatas = [
    {"category": "فناوری", "topic": "AI", "source": "دانشنامه کامپیوتر"},
    {"category": "فناوری", "topic": "Deep Learning", "source": "مقاله تخصصی"},
    {"category": "طبیعت", "topic": "گیاهان", "source": "سازمان جنگل‌ها"},
    {"category": "طبیعت", "topic": "زیست‌شناسی", "source": "کتاب علوم"},
    {"category": "ورزش", "topic": "فوتبال", "source": "خبرگزاری ورزش"},
    {"category": "ورزش", "topic": "سلامت", "source": "مجله تندرستی"},
    {"category": "فناوری", "topic": "امنیت", "source": "امنیت شبکه"},
    {"category": "فناوری", "topic": "دیتابیس", "source": "کوئرا کالج"}
]

print(f"تعداد کل اسناد: {len(documents)}")

تعداد کل اسناد: 8


### ۷. ذخیره اسناد در پایگاه برداری Chroma و اندازه‌گیری زمان ساخت ایندکس (Write Latency)

همان‌طور که در تئوری بررسی کردیم، در زمان اضافه کردن اسناد به دیتابیس برداری، دو کار سنگین انجام می‌شود:
1. تولید بردارهای تعبیه (ارسال متن‌ها به مدل Embedding).
2. ساخت گراف چندلایه HNSW و اتصال بردارها به گره‌های مجاور (Write Cost).

این کار باعث می‌شود زمان نوشتن کمی طولانی‌تر باشد، اما در عوض زمان خواندن و جستجو **آنی و میلی‌ثانیه‌ای** خواهد بود!

In [3]:
from langchain_chroma import Chroma

print("در حال ساخت وکتور دیتابیس و ایندکس‌گذاری HNSW...")
start_write = time.time()

# ساخت پایگاه داده برداری در حافظه موقت (می‌توان با persist_directory آن را روی دیسک پایدار کرد)
vector_db = Chroma.from_texts(
    texts=documents,
    metadatas=metadatas,
    embedding=embedding_model,
    collection_name="tech_and_nature_collection"
)

write_duration = time.time() - start_write
print(f"✅ ایندکس‌گذاری برداری با موفقیت انجام شد! زمان صرف‌شده: {write_duration:.2f} ثانیه")

در حال ساخت وکتور دیتابیس و ایندکس‌گذاری HNSW...
✅ ایندکس‌گذاری برداری با موفقیت انجام شد! زمان صرف‌شده: 0.99 ثانیه


### ۸. اجرای جستجوی شباهت معنایی (Semantic Search) و اندازه‌گیری زمان پاسخ (Read Latency)

اکنون سوالی مطرح می‌کنیم که کلمات کلیدی آن عیناً در اسناد وجود ندارد؛ مثلاً «چگونه کامپیوترها مانند انسان فکر می‌کنند؟».
سپس سرعت پاسخگویی دیتابیس برداری را بررسی می‌کنیم.

In [4]:
query = "چگونه کامپیوترها شبیه انسان مسائل را یاد می‌گیرند؟"

start_read = time.time()
# جستجوی ۲ سند برتر
results = vector_db.similarity_search(query, k=2)
read_duration = time.time() - start_read

print(f"🔍 جستجو برای سوال: '{query}'")
print(f"⏱️ زمان اجرای جستجوی ایندکس: {read_duration * 1000:.2f} میلی‌ثانیه\n")

for idx, doc in enumerate(results, 1):
    print(f"نتیجه {idx}:")
    print(f"  متن: {doc.page_content}")
    print(f"  دسته‌بندی: {doc.metadata.get('category')} | موضوع: {doc.metadata.get('topic')}")
    print(f"  منبع: {doc.metadata.get('source')}\n")

🔍 جستجو برای سوال: 'چگونه کامپیوترها شبیه انسان مسائل را یاد می‌گیرند؟'
⏱️ زمان اجرای جستجوی ایندکس: 150.92 میلی‌ثانیه

نتیجه 1:
  متن: هوش مصنوعی و یادگیری ماشین شاخه‌ای از علوم کامپیوتر هستند که سیستم‌ها را قادر می‌سازند از داده‌ها بیاموزند.
  دسته‌بندی: فناوری | موضوع: AI
  منبع: دانشنامه کامپیوتر

نتیجه 2:
  متن: شبکه‌های عصبی عمیق با لایه‌های پنهان متعدد برای پردازش زبان طبیعی و بینایی ماشین استفاده می‌شوند.
  دسته‌بندی: فناوری | موضوع: Deep Learning
  منبع: مقاله تخصصی



### ۹. تحلیل امتیاز شباهت و فاصله (Similarity Search with Score)

متد `similarity_search_with_score` علاوه بر متن سند، فاصله برداری را نیز برمی‌گرداند.

> **نکته مهم:** در ChromaDB به طور پیش‌فرض از معیار فاصله **فاصله مربعی اقلیدسی ($L_2$ Distance)** یا فاصله کسینوسی معکوس استفاده می‌شود. بنابراین:
> - **امتیاز کوچکتر (نزدیک‌تر به صفر) = شباهت بیشتر و فاصله کمتر.**
> - **امتیاز بزرگتر = تفاوت معنایی بیشتر.**

In [5]:
query_score = "روش نگهداری و محافظت از پوشش گیاهی و درختان"

results_with_scores = vector_db.similarity_search_with_score(query_score, k=3)

print(f"🔍 سوال: '{query_score}'\n")
for idx, (doc, score) in enumerate(results_with_scores, 1):
    print(f"رتبه {idx} [فاصله برداری: {score:.4f}]:")
    print(f"  متن: {doc.page_content}")
    print(f"  دسته: {doc.metadata.get('category')} ({doc.metadata.get('topic')})\n")

🔍 سوال: 'روش نگهداری و محافظت از پوشش گیاهی و درختان'

رتبه 1 [فاصله برداری: 15.2336]:
  متن: درختان بلوط و راش از گونه‌های مهم جنگل‌های هیرکانی در شمال ایران هستند.
  دسته: طبیعت (گیاهان)

رتبه 2 [فاصله برداری: 19.5288]:
  متن: گیاهان از طریق فتوسنتز، دی‌اکسید کربن هوا را جذب کرده و اکسیژن و قند تولید می‌کنند.
  دسته: طبیعت (زیست‌شناسی)

رتبه 3 [فاصله برداری: 24.1962]:
  متن: تیم ملی فوتبال آرژانتین با درخشش لیونل مسی قهرمان جام جهانی فوتبال در قطر شد.
  دسته: ورزش (فوتبال)



### ۱۰. فیلترینگ ترکیبی: فیلتر فراداده (Metadata Filter) + جستجوی معنایی

یکی از مزیت‌های کلیدی لایه ذخیره‌سازی وکتور دیتابیس‌ها، امکان اعمال فیلتر بر روی فراداده‌ها (مانند SQL WHERE clause) همزمان با جستجوی برداری است.

برای نمونه، فرض کنید می‌خواهیم سوال ورزشی بپرسیم و مطمئن باشیم که فقط در اسناد دسته‌بندی «ورزش» جستجو می‌شود:

In [6]:
sports_query = "بهترین راه برای حفظ انرژی و شادابی بدن چیست؟"

# اعمال فیلتر فراداده
filtered_results = vector_db.similarity_search(
    sports_query,
    k=2,
    filter={"category": "ورزش"}
)

print(f"🔍 جستجو با فیلتر category='ورزش' برای سوال: '{sports_query}'\n")
for idx, doc in enumerate(filtered_results, 1):
    print(f"سند {idx}:")
    print(f"  متن: {doc.page_content}")
    print(f"  دسته: {doc.metadata.get('category')} | موضوع: {doc.metadata.get('topic')}\n")

🔍 جستجو با فیلتر category='ورزش' برای سوال: 'بهترین راه برای حفظ انرژی و شادابی بدن چیست؟'

سند 1:
  متن: ورزش روزانه و تمرینات هوازی سلامت قلب و عروق و طول عمر انسان را بهبود می‌بخشند.
  دسته: ورزش | موضوع: سلامت

سند 2:
  متن: تیم ملی فوتبال آرژانتین با درخشش لیونل مسی قهرمان جام جهانی فوتبال در قطر شد.
  دسته: ورزش | موضوع: فوتبال



### 🎯 جمع‌بندی نکات کلیدی این بخش:

1. **آناتومی وکتور دیتابیس:** شامل دو بخش اصلی **Storage Layer** (برای دوام و نگهداری متن و فراداده روی دیسک) و **Index Layer** (ساختار داده‌های سریع HNSW / IVF در RAM برای ناوبری سریع).
2. **جستجوی تقریبی (ANN):** با فدا کردن درصد بسیار ناچیزی از دقت، سرعت جستجو را از $O(N)$ به $O(\log N)$ تبدیل می‌کند.
3. **الگوریتم‌های ایندکس:**
   - **IVF:** مناسب برای بهینه‌سازی مصرف حافظه با استفاده از خوشه‌بندی سنترویدها.
   - **HNSW:** گراف چندلایه‌ای پیشرفته با بالاترین سرعت و Recall (استاندارد مدرن پایگاه‌های برداری).
4. **معاوضه سرعت/دقت/حافظه:** هزینه اصلی الگوریتم HNSW، مصرف حافظه رم بیشتر و زمان پردازش در هنگام ثبت اسناد (Write Latency) است که در ازای آن، سرعت جستجوی زیر ۱۰ میلی‌ثانیه در زمان استعلام (Read Latency) حاصل می‌شود.